<a href="https://colab.research.google.com/github/aranagnost/drone-audio-classification/blob/main/colab_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Drone Audio Classification — Colab Training

**Before running:** Runtime > Change runtime type > T4 GPU

## 1. Mount Google Drive

In [7]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. Clone the repo

In [8]:
import os
import sys

REPO_DIR = '/content/drone-audio-classification'

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/aranagnost/drone-audio-classification.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print('Working directory:', os.getcwd())

Already up to date.
Working directory: /content/drone-audio-classification


## 3. Verify GPU and dataset

In [15]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

DATASET_ROOT = '/content/drive/MyDrive/drone-audio/datasets/Drone_Audio_Dataset/audio'
print('\nDataset root exists:', os.path.exists(DATASET_ROOT))
print('Subdirs:', os.listdir(DATASET_ROOT))

CUDA available: True
GPU: Tesla T4

Dataset root exists: True
Subdirs: ['6_motors', 'not_a_drone', '2_motors', '8_motors', '4_motors']


## 4. Create artifacts directory

In [16]:
os.makedirs('artifacts/checkpoints', exist_ok=True)

## 5. Load the dataset

In [ ]:
import shutil
print("Copying dataset to local disk...")
shutil.copytree(
    '/content/drive/MyDrive/drone-audio/datasets/',
    '/content/datasets/'
)
print("Done.")

Copying dataset to local disk...


## 6. Train Stage 1 (binary: drone vs no-drone)

In [11]:
MODEL = 'small_cnn_v2'  # options: small_cnn_v1, small_cnn_v2, small_cnn_v3, big_cnn_v1, 3_stages_cnn_v1

!PYTHONPATH={REPO_DIR} python training/train_stage1.py \
    --model {MODEL} \
    --dataset_root {DATASET_ROOT} \
    --num_workers 2

[INFO] device=cuda  model=small_cnn_v2 (SmallCNNv2)  dropout=0.2
[INFO] SpecAugment: freq_mask=8, time_mask=16
[INFO] Parameters: 79,790
object address  : 0x7999990835e0
object refcount : 3
object type     : 0xa284e0
object type name: KeyboardInterrupt
object repr     : KeyboardInterrupt()
lost sys.stderr


## 7. Train Stage 2 (motor count: 2/4/6/8 motors)

In [12]:
!PYTHONPATH={REPO_DIR} python training/train_stage2.py \
    --model {MODEL} \
    --dataset_root {DATASET_ROOT} \
    --num_workers 2

[INFO] device=cuda  model=small_cnn_v2 (SmallCNNv2)  task=stage2  dropout=0.2
[INFO] Classes: ['2_motors', '4_motors', '6_motors', '8_motors']
[INFO] Train samples: 5757  Val samples: 734
[INFO] SpecAugment: freq_mask=8, time_mask=16
[INFO] Parameters: 79,856
object address  : 0x7b6b67953d00
object refcount : 3
object type     : 0xa284e0
object type name: KeyboardInterrupt
object repr     : KeyboardInterrupt()
lost sys.stderr


## 8. Evaluate Stage 1 + Stage 2

In [13]:
!PYTHONPATH={REPO_DIR} python training/eval.py \
    --model {MODEL} \
    --dataset_root {DATASET_ROOT}

Traceback (most recent call last):
object address  : 0x7ae348cc6da0
object refcount : 3
object type     : 0xa284e0
object type name: KeyboardInterrupt
object repr     : KeyboardInterrupt()
lost sys.stderr
^C


## 9. Copy checkpoints to Drive for persistence

Colab resets between sessions — save checkpoints to Drive so you don't lose them.

In [14]:
import shutil

CKPT_DEST = f'/content/drive/MyDrive/drone-audio/checkpoints'
os.makedirs(CKPT_DEST, exist_ok=True)
shutil.copytree('artifacts/checkpoints', CKPT_DEST, dirs_exist_ok=True)
print('Checkpoints saved to:', CKPT_DEST)

Checkpoints saved to: /content/drive/MyDrive/drone-audio/checkpoints
